# AgentCore Runtime + Gateway + MCP + A2A: a guided, self-contained tour

**What you will build** (everything runs on your laptop, no AWS account, no LLM key, no database):

```
                         +--------------------------------------------+
  you (notebook) ------> |  SUPERVISOR  (AgentCore Runtime app :8090) |
   POST /invocations     |  1. routes the message                     |
                         +-------------+------------------+-----------+
                                       |                  |
                       "order question"|                  |"where is my parcel?"
                                       v                  v
                         +-------------------+     +--------------------------+
                         |  ORDER AGENT      |     | SHIPMENT TRACKING AGENT  |
                         |  (inside the      |     | separate service :9000   |
                         |   supervisor)     |     | speaks A2A               |
                         +---------+---------+     +--------------------------+
                                   | MCP (tools)
                                   v
                         +-------------------+
                         | ORDER MCP SERVER  |   <-- in AWS, AgentCore GATEWAY sits in front of this
                         | :8001  (3 tools)  |
                         +-------------------+
```

**The four ideas, in one sentence each**

| Idea | One-liner | Where in this notebook |
|---|---|---|
| **MCP** (Model Context Protocol) | A standard way for an agent to *discover and call tools* on a server. | Part 2, 3 |
| **A2A** (Agent2Agent) | A standard way for one *agent to call another agent* (even built by someone else). | Part 4 |
| **AgentCore Runtime** | The managed place where your agent code runs: it expects `POST /invocations` and `GET /ping`. | Part 5 |
| **AgentCore Gateway** | A managed front door that turns APIs / MCP servers into *one authenticated MCP endpoint* for agents. | Part 6 |

**Cost note:** Parts 1-5 are local and free. Part 6 (Gateway) needs AWS, so its code is guarded by a flag that is
`False` by default. Nothing in this notebook touches AWS unless you flip it.

**Before you start:** select the project's `.venv` as the notebook kernel (it needs `ipykernel`; VS Code offers to install it).

# Part 0. Setup

## 0.1 Imports and logging

We use `logging` for output (a project convention). `force=True` makes it work even though Jupyter has already
configured logging. The `show()` helper just keeps the later cells short.

In [1]:
import asyncio
import json
import logging
import socket
import subprocess
import sys
import time
from pathlib import Path

import httpx

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
logger = logging.getLogger("notebook")
for noisy in ("httpx", "mcp", "a2a", "httpcore"):   # hide per-request HTTP chatter
    logging.getLogger(noisy).setLevel(logging.WARNING)


def show(title: str, value: object = "") -> None:
    """Log a titled result so notebook output stays readable.

    Args:
        title: Short label for the line.
        value: Anything printable that belongs to the label.
    """
    logger.info("%s %s", title, value)

## 0.2 Check the libraries are installed

These are already dependencies of this repo. If one import fails you are most likely on the wrong kernel.

In [2]:
import importlib.metadata as md_

for package in ["mcp", "a2a-sdk", "bedrock-agentcore", "uvicorn", "httpx", "boto3"]:
    try:
        show(f"{package:20}", md_.version(package))
    except md_.PackageNotFoundError:
        show(f"{package:20}", "MISSING -> pick the project .venv kernel (boto3 is only needed for Part 6)")

mcp                  1.30.0
a2a-sdk              1.2.2
bedrock-agentcore    1.25.0
uvicorn              0.54.0
httpx                0.28.1
boto3                1.43.118


## 0.3 A folder for the three small services, and helpers to start/stop them

Each service is a normal Python file that we write with `%%writefile` **from this notebook**, so all the code is
visible here. We then start each one as a background process (`subprocess`) and wait until its port answers.

Why separate processes? Because that is exactly how it works in production: the MCP server, the shipment agent and
the supervisor are *three independent deployments* that only talk over HTTP.

In [3]:
DEMO_DIR = Path("agentcore_demo")
DEMO_DIR.mkdir(exist_ok=True)

processes: dict[str, subprocess.Popen] = {}   # service name -> running process


def port_open(port: int) -> bool:
    """Check whether something is listening on a local port.

    Args:
        port: TCP port on 127.0.0.1.

    Returns:
        bool: True if a connection succeeds.
    """
    with socket.socket() as s:
        s.settimeout(0.3)
        return s.connect_ex(("127.0.0.1", port)) == 0


def start_service(name: str, script: str, port: int, env: dict[str, str] | None = None) -> None:
    """Start a python file as a background process and wait until its port is open.

    Args:
        name: Label used for the log file and the `processes` registry.
        script: File inside `agentcore_demo/` to run.
        port: Port the service will listen on.
        env: Extra environment variables for the process (optional).

    Raises:
        RuntimeError: If the port is not open after 20 seconds (the log tail is included).
    """
    import os

    stop_service(name)
    for _ in range(20):                      # give a just-stopped service a moment to release its port
        if not port_open(port):
            break
        time.sleep(0.25)
    if port_open(port):
        raise RuntimeError(f"Port {port} is already in use by another program. Stop it or change the port.")
    log_path = DEMO_DIR / f"{name}.log"
    process = subprocess.Popen(
        [sys.executable, "-u", str(DEMO_DIR / script)],
        stdout=open(log_path, "w"), stderr=subprocess.STDOUT,
        env={**os.environ, **(env or {})},
    )
    processes[name] = process
    for _ in range(40):
        if port_open(port):
            show(f"started {name}", f"-> http://127.0.0.1:{port}  (log: {log_path})")
            return
        time.sleep(0.5)
    raise RuntimeError(f"{name} did not start. Log:\n{log_path.read_text()[-1500:]}")


def stop_service(name: str) -> None:
    """Stop a service started by `start_service` (no-op if it is not running).

    Args:
        name: Label used when the service was started.
    """
    process = processes.pop(name, None)
    if process and process.poll() is None:
        process.terminate()
        process.wait(timeout=10)

# Part 1. The fake data

To stay self-contained we replace the real database with a tiny Python dictionary that lives *inside* the MCP server
file (next part). Two customers, a few orders each. Order ids look like `ORD-1001`.

Nothing to run here; it is explained so the next cell is not a surprise.

# Part 2. MCP server: give agents tools

## What is MCP?

MCP is a protocol with two roles:

- **MCP server**: exposes *tools* (functions) with a name, a description and an input schema.
- **MCP client** (inside an agent): asks `list_tools()` to *discover* them, then `call_tool(name, args)`.

The agent does not need to know the server's code. It only needs the URL. That is also why **AgentCore Gateway**
can later sit in the middle: it is just another MCP endpoint that forwards your calls.

We use the **streamable HTTP** transport (a normal HTTP server at `/mcp`), because that is what a remote server
(and Gateway) needs. The other option, `stdio`, only works for a child process on the same machine.

## 2.1 Write the order MCP server

`FastMCP` turns every function decorated with `@mcp.tool()` into a tool. The **docstring becomes the tool
description** the agent reads, so write it for the agent.

In [ ]:
%%writefile agentcore_demo/order_mcp_server.py
"""Order MCP server: three read-only tools over an in-memory order table (demo data)."""

import logging
import sys

from mcp.server.fastmcp import FastMCP

logging.basicConfig(level=logging.INFO, stream=sys.stderr, format="[order-mcp] %(message)s")
logger = logging.getLogger(__name__)

# Fake "database": customer id -> orders (newest first).
ORDERS: dict[str, list[dict]] = {
    "cust001": [
        {"order_id": "ORD-1002", "date": "2024-09-14", "channel": "online", "items": ["Linen shirt", "Straight jeans"]},
        {"order_id": "ORD-1001", "date": "2024-06-02", "channel": "store", "items": ["Wool coat"]},
    ],
    "cust002": [
        {"order_id": "ORD-2001", "date": "2024-08-21", "channel": "online", "items": ["Running shorts"]},
    ],
}

mcp = FastMCP("order-management", host="127.0.0.1", port=8001)  # streamable HTTP served at /mcp


@mcp.tool()
def list_orders(customer_id: str) -> str:
    """List all orders of a customer (id, date, channel, number of items), newest first.

    Args:
        customer_id: Customer id such as cust001.
    """
    orders = ORDERS.get(customer_id, [])
    logger.info("tool=list_orders orders=%d", len(orders))
    if not orders:
        return "No orders found for this customer."
    return "\n".join(f"{o['order_id']}: {o['date']} ({o['channel']}), {len(o['items'])} item(s)" for o in orders)


@mcp.tool()
def get_order_details(customer_id: str, order_id: str) -> str:
    """Get the items of one order. Only finds orders that belong to the given customer.

    Args:
        customer_id: Customer id such as cust001.
        order_id: Order id such as ORD-1002.
    """
    for order in ORDERS.get(customer_id, []):
        if order["order_id"] == order_id.upper():
            return f"Order {order['order_id']} placed on {order['date']} ({order['channel']}): {', '.join(order['items'])}."
    return "No order with that id was found for this customer."


@mcp.tool()
def get_last_bought_product(customer_id: str) -> str:
    """Get the name of the last product a customer bought.

    Args:
        customer_id: Customer id such as cust001.
    """
    orders = ORDERS.get(customer_id, [])
    return orders[0]["items"][-1] if orders else "No purchases found for this customer."


if __name__ == "__main__":
    mcp.run(transport="streamable-http")

## 2.2 Start the MCP server

In [ ]:
start_service("order_mcp", "order_mcp_server.py", port=8001)

## 2.3 Talk to it like an agent would: discover tools, then call one

This is the **entire MCP client side** in a few lines. Jupyter allows `await` at the top level of a cell.

1. open a connection to the URL,
2. `initialize()` the session (handshake),
3. `list_tools()` to see what exists,
4. `call_tool()` to use one.

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

ORDER_MCP_URL = "http://127.0.0.1:8001/mcp"

async with streamablehttp_client(ORDER_MCP_URL) as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()

        tools = await session.list_tools()
        for tool in tools.tools:
            show(f"tool: {tool.name}", f"- {tool.description.splitlines()[0]}")
            show("   input schema:", json.dumps(tool.inputSchema["properties"]))

        result = await session.call_tool("list_orders", {"customer_id": "cust001"})
        show("\ncall_tool(list_orders) ->")
        show(result.content[0].text)

> **Take-away:** the agent only sees *names, descriptions and schemas*. In a real LLM agent, those schemas are handed to
> the model, which picks a tool and fills the arguments. In this notebook we pick the tool with simple rules (Part 3),
> so you can read and run everything without an API key. Swapping in an LLM changes only the "pick a tool" step.

# Part 3. The order agent (an MCP client)

An *agent* here = **decide what to do** + **call tools** + **write the answer**.

Our order agent does this:

1. connect to the MCP server and `list_tools()`,
2. *decide* which tool fits the question (rules; an LLM would do this in production),
3. call it and return the text.

One more important detail, **who is the customer?** The tools take a `customer_id`. In production the system
(from the verified login token) supplies it, *never the LLM*, otherwise a user could ask for someone else's data.
Here, `customer_id` is a plain function argument that the caller decides.

The agent takes the MCP URL as a parameter. Keep that in mind for Part 6: **pointing it at Gateway is just a different URL**.

In [ ]:
import re

ORDER_ID_PATTERN = re.compile(r"\bORD-\d+\b", re.IGNORECASE)


def choose_order_tool(message: str, available: set[str]) -> tuple[str, dict]:
    """Pick the MCP tool (and its extra arguments) for an order question.

    This is the "brain" of the demo agent. A real agent would give the tool list to an LLM instead.

    Args:
        message: The customer's message.
        available: Tool names the server reported via `list_tools()`.

    Returns:
        tuple[str, dict]: Tool name and arguments other than `customer_id`.
    """
    order_id = ORDER_ID_PATTERN.search(message)
    text = message.lower()
    if order_id and "get_order_details" in available:
        return "get_order_details", {"order_id": order_id.group(0).upper()}
    if ("last" in text or "latest" in text) and "get_last_bought_product" in available:
        return "get_last_bought_product", {}
    return "list_orders", {}


async def order_agent(message: str, customer_id: str, mcp_url: str = ORDER_MCP_URL) -> str:
    """Answer an order question by calling a tool on the MCP server.

    Args:
        message: The customer's message.
        customer_id: Customer the data belongs to (set by the system, not by the model).
        mcp_url: URL of the MCP endpoint (the MCP server directly, or an AgentCore Gateway).

    Returns:
        str: The tool's answer text.
    """
    async with streamablehttp_client(mcp_url) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            available = {t.name for t in (await session.list_tools()).tools}
            tool_name, extra_args = choose_order_tool(message, available)
            logger.info("order_agent: chose tool=%s", tool_name)
            result = await session.call_tool(tool_name, {"customer_id": customer_id, **extra_args})
            return result.content[0].text


for question in ["What did I order?", "What was my last purchase?", "What is in order ORD-1002?"]:
    show(f"Q: {question}")
    show("A:", await order_agent(question, "cust001"))

# Part 4. Shipment tracking agent as an A2A service

## What is A2A?

A2A lets an agent be a **service that other agents can call**, without sharing code. Three things to know:

| Concept | Meaning |
|---|---|
| **Agent Card** | A JSON "business card" served at `/.well-known/agent-card.json`: name, URL, skills, and capabilities such as streaming. Clients read it first (discovery). |
| **Task** | One unit of work. It moves through states: `submitted` -> `working` -> `completed` (or `failed`). |
| **Artifact** | The result of a task (here: the reply text). Streaming sends it in pieces. |

Why A2A for shipment tracking? The carrier lookup is **slow**. With A2A the agent can report `working` status updates
("Contacting the carrier...") while it waits, and the caller can stream them to the user.

## 4.1 Write the shipment A2A server

Read it top to bottom:

1. `build_card()` describes the agent (the Agent Card).
2. `ShipmentExecutor.execute()` is the work. It publishes events to `updater`; the SDK turns them into the HTTP stream.
3. The last lines wire the card + executor into a web app and start it on **port 9000**, the port AgentCore Runtime
   expects for A2A agents.

The "carrier" is simulated: it sleeps briefly and returns a status picked from the order id.

In [ ]:
%%writefile agentcore_demo/shipment_a2a_server.py
"""Shipment tracking agent exposed over A2A (simulated carrier lookup)."""

import asyncio
import hashlib
import logging
import re

import uvicorn
from a2a.helpers.proto_helpers import new_task_from_user_message, new_text_message, new_text_part
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events.event_queue_v2 import EventQueue
from a2a.server.request_handlers import DefaultRequestHandlerV2
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore, TaskUpdater
from a2a.types.a2a_pb2 import AgentCapabilities, AgentCard, AgentInterface, AgentSkill, TaskState
from a2a.utils.constants import AGENT_CARD_WELL_KNOWN_PATH, PROTOCOL_VERSION_CURRENT, TransportProtocol
from starlette.applications import Starlette

logging.basicConfig(level=logging.INFO, format="[shipment-a2a] %(message)s")
logger = logging.getLogger(__name__)

PORT = 9000
ORDER_ID_PATTERN = re.compile(r"\bORD-\d+\b", re.IGNORECASE)
CARRIERS = [
    ("DHL Express", "Hamburg sorting hub", "In transit, expected delivery in 2 days"),
    ("PostNord", "Stockholm distribution centre", "Out for delivery, expected today"),
    ("UPS", "Rotterdam depot", "Held at the depot, delivery attempt tomorrow"),
]


def build_card() -> AgentCard:
    """Build the Agent Card: the public self-description clients read before talking to us.

    Returns:
        AgentCard: Card with one skill; `streaming=True` tells clients they can watch progress live.
    """
    return AgentCard(
        name="shipment_tracking",
        description="Checks the carrier for the shipping status of an order. The lookup is slow, so it streams progress.",
        version="1.0.0",
        supported_interfaces=[
            AgentInterface(
                url=f"http://127.0.0.1:{PORT}",
                protocol_binding=TransportProtocol.JSONRPC,
                protocol_version=PROTOCOL_VERSION_CURRENT,
            )
        ],
        capabilities=AgentCapabilities(streaming=True),
        default_input_modes=["text/plain"],
        default_output_modes=["text/plain"],
        skills=[
            AgentSkill(
                id="shipment-status", name="Shipment status lookup",
                description="Carrier, last scan and delivery outlook for an order id.",
                tags=["shipping"], examples=["Where is my order ORD-1002?"],
            )
        ],
    )


class ShipmentExecutor(AgentExecutor):
    """The agent's work: turn one incoming message into a streamed task."""

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        """Run one task: announce it, report progress, publish the answer, complete.

        Args:
            context: Incoming message plus task/context ids and request metadata.
            event_queue: Where task events are published (the SDK streams them to the caller).
        """
        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        if context.current_task is None:
            await event_queue.enqueue_event(new_task_from_user_message(context.message))  # state: submitted
        await updater.start_work()                                                         # state: working

        match = ORDER_ID_PATTERN.search(context.get_user_input())
        if not match:
            await updater.add_artifact([new_text_part("Which order should I track? Send an id like ORD-1002.")],
                                       artifact_id="reply", name="reply", last_chunk=True)
            await updater.complete()
            return

        order_id = match.group(0).upper()
        logger.info("task=%s order=%s status=start", context.task_id, order_id)
        for step in ["Contacting the carrier", "Waiting for the carrier system", "Reading the latest scan events"]:
            await updater.update_status(TaskState.TASK_STATE_WORKING, new_text_message(step))  # progress line
            await asyncio.sleep(1)                                                             # the "slow" part

        digest = int(hashlib.sha256(order_id.encode()).hexdigest()[:8], 16)
        carrier, location, outlook = CARRIERS[digest % len(CARRIERS)]
        reply = f"Order {order_id} is with {carrier}. Last scan: {location}. {outlook}."
        await updater.add_artifact([new_text_part(reply)], artifact_id="reply", name="reply", last_chunk=True)
        await updater.complete()                                                               # state: completed
        logger.info("task=%s status=complete", context.task_id)

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        """Mark the task cancelled (our runs are short, so there is nothing to stop).

        Args:
            context: Request context holding the task to cancel.
            event_queue: Queue the cancellation event is published to.
        """
        await TaskUpdater(event_queue, context.task_id, context.context_id).cancel()


card = build_card()
handler = DefaultRequestHandlerV2(agent_executor=ShipmentExecutor(), task_store=InMemoryTaskStore(), agent_card=card)
app = Starlette(routes=[
    *create_agent_card_routes(card, card_url=AGENT_CARD_WELL_KNOWN_PATH),  # GET /.well-known/agent-card.json
    *create_jsonrpc_routes(handler, rpc_url="/"),                          # POST /  (JSON-RPC messages)
])

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning")

## 4.2 Start it and read its Agent Card (discovery)

In [ ]:
start_service("shipment_a2a", "shipment_a2a_server.py", port=9000)

card_json = httpx.get("http://127.0.0.1:9000/.well-known/agent-card.json").json()
show("Agent Card:")
show(json.dumps(card_json, indent=2)[:1200])

## 4.3 Call it as an A2A client, with streaming

The client does what any A2A caller does:

1. fetch the card (`A2ACardResolver`),
2. build a client from the card (`ClientFactory`), which uses streaming if the card says so,
3. send a `SendMessageRequest`,
4. read the **stream of responses**: a `task` (created), `status_update`s (progress and state), `artifact_update`s (answer pieces).

We wrap this in an async generator `ask_shipment_agent` that yields `("progress" | "answer" | "state", text)` tuples.
The supervisor in Part 5 will reuse the exact same idea.

In [ ]:
from collections.abc import AsyncIterator

from a2a.client import A2ACardResolver, ClientConfig, ClientFactory
from a2a.helpers.proto_helpers import get_artifact_text, get_message_text, new_text_part
from a2a.types.a2a_pb2 import Message, Role, SendMessageRequest, TaskState

SHIPMENT_A2A_URL = "http://127.0.0.1:9000"


async def ask_shipment_agent(text: str, customer_id: str, base_url: str = SHIPMENT_A2A_URL) -> AsyncIterator[tuple[str, str]]:
    """Send a message to an A2A agent and yield what it streams back.

    Args:
        text: The customer's message.
        customer_id: Customer id, sent as request metadata (the agent could use it to check ownership).
        base_url: Base URL of the A2A agent.

    Yields:
        tuple[str, str]: `("progress", line)`, `("answer", piece)` or `("state", final_task_state)`.
    """
    async with httpx.AsyncClient(timeout=60) as http:
        card = await A2ACardResolver(http, base_url).get_agent_card()          # 1. discover
        client = ClientFactory(ClientConfig(streaming=card.capabilities.streaming, httpx_client=http)).create(card)
        request = SendMessageRequest(                                          # 3. the message
            message=Message(message_id=f"msg-{time.time_ns()}", role=Role.ROLE_USER, parts=[new_text_part(text)]),
            metadata={"customer_id": customer_id},
        )
        async for response in client.send_message(request):                    # 4. the stream
            if response.HasField("status_update"):
                status = response.status_update.status
                if status.state == TaskState.TASK_STATE_WORKING and status.message.parts:
                    yield "progress", get_message_text(status.message)
                elif status.state == TaskState.TASK_STATE_COMPLETED:
                    yield "state", TaskState.Name(status.state)
            elif response.HasField("artifact_update"):
                yield "answer", get_artifact_text(response.artifact_update.artifact, delimiter="")
        await client.close()


async for kind, value in ask_shipment_agent("Where is my order ORD-1002?", "cust001"):
    show(f"[{kind:8}]", value)

> **MCP vs A2A, the difference to remember:** MCP = *agent -> tool* (a function call with a schema).
> A2A = *agent -> agent* (a conversation-like **task** with status updates and streamed results).

# Part 5. The supervisor on AgentCore Runtime

## What is AgentCore Runtime?

Runtime hosts *your* agent code (as a container) and gives it serverless scaling, session isolation and auth.
Your code only has to follow a small **contract**:

| Contract | Meaning |
|---|---|
| `GET /ping` | Health check. |
| `POST /invocations` (port **8080** in AWS; we use 8090 locally in case 8080 is taken) | One request in, one (or a streamed) response out. |
| Session id header `X-Amzn-Bedrock-AgentCore-Runtime-Session-Id` | Identifies one conversation. |

The `bedrock-agentcore` SDK gives you that contract with a decorator: `BedrockAgentCoreApp` + `@app.entrypoint`.
Runtime can also host other shapes: **MCP servers** (port 8000, `/mcp`) and **A2A agents** (port 9000), which is how
our MCP server and shipment agent would be deployed too.

## 5.1 Write the supervisor

It does three things per message:

1. **Route**: decide `order_management`, `shipment_tracking` or `out_of_scope` (keyword rules here; the real project asks a small LLM).
2. **Delegate**: order questions -> MCP tools (in-process agent), shipping questions -> A2A call to the other service.
3. **Stream events** back by `yield`ing dicts. Because the entrypoint is an `async` generator, Runtime streams them as they happen.

The two URLs come from environment variables. That is how you will repoint them to AWS later without editing code.

In [ ]:
%%writefile agentcore_demo/supervisor_runtime.py
"""Supervisor as an AgentCore Runtime app: routes a message to the order agent (MCP) or shipment agent (A2A)."""

import logging
import os
import re
import time

import httpx
from a2a.client import A2ACardResolver, ClientConfig, ClientFactory
from a2a.helpers.proto_helpers import get_artifact_text, get_message_text, new_text_part
from a2a.types.a2a_pb2 import Message, Role, SendMessageRequest, TaskState
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

logging.basicConfig(level=logging.INFO, format="[supervisor] %(message)s")
logger = logging.getLogger(__name__)

# Where the other pieces live. Locally: direct URLs. In AWS: the Gateway URL / the A2A runtime URL.
ORDER_MCP_URL = os.environ.get("ORDER_MCP_URL", "http://127.0.0.1:8001/mcp")
ORDER_MCP_TOKEN = os.environ.get("ORDER_MCP_TOKEN", "")  # bearer token, needed when the URL is a Gateway
SHIPMENT_A2A_URL = os.environ.get("SHIPMENT_A2A_URL", "http://127.0.0.1:9000")
ORDER_ID_PATTERN = re.compile(r"\bORD-\d+\b", re.IGNORECASE)

app = BedrockAgentCoreApp()  # provides GET /ping and POST /invocations


def route(message: str) -> str:
    """Decide which agent should handle the message (the supervisor's only decision).

    Args:
        message: The customer's message.

    Returns:
        str: `shipment_tracking`, `order_management` or `out_of_scope`.
    """
    text = message.lower()
    if any(word in text for word in ("where", "track", "deliver", "parcel", "shipping", "shipment")):
        return "shipment_tracking"
    if any(word in text for word in ("order", "bought", "buy", "purchase")):
        return "order_management"
    return "out_of_scope"


async def run_order_agent(message: str, customer_id: str) -> str:
    """Order agent: pick an MCP tool by simple rules, call it, return its text.

    Args:
        message: The customer's message.
        customer_id: Customer id supplied by the system (never by the model).

    Returns:
        str: Tool result text.
    """
    headers = {"Authorization": f"Bearer {ORDER_MCP_TOKEN}"} if ORDER_MCP_TOKEN else None
    async with streamablehttp_client(ORDER_MCP_URL, headers=headers) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            order_id = ORDER_ID_PATTERN.search(message)
            if order_id:
                tool, args = "get_order_details", {"order_id": order_id.group(0).upper()}
            elif "last" in message.lower():
                tool, args = "get_last_bought_product", {}
            else:
                tool, args = "list_orders", {}
            logger.info("order_agent tool=%s", tool)
            result = await session.call_tool(tool, {"customer_id": customer_id, **args})
            return result.content[0].text


async def run_shipment_agent(message: str, customer_id: str):
    """Call the shipment agent over A2A and yield its progress lines and answer pieces.

    Args:
        message: The customer's message.
        customer_id: Customer id, forwarded as request metadata.

    Yields:
        tuple[str, str]: `("progress", text)` or `("answer", text)`.
    """
    async with httpx.AsyncClient(timeout=60) as http:
        card = await A2ACardResolver(http, SHIPMENT_A2A_URL).get_agent_card()
        client = ClientFactory(ClientConfig(streaming=card.capabilities.streaming, httpx_client=http)).create(card)
        request = SendMessageRequest(
            message=Message(message_id=f"msg-{time.time_ns()}", role=Role.ROLE_USER, parts=[new_text_part(message)]),
            metadata={"customer_id": customer_id},
        )
        async for response in client.send_message(request):
            if response.HasField("status_update"):
                status = response.status_update.status
                if status.state == TaskState.TASK_STATE_WORKING and status.message.parts:
                    yield "progress", get_message_text(status.message)
            elif response.HasField("artifact_update"):
                yield "answer", get_artifact_text(response.artifact_update.artifact, delimiter="")
        await client.close()


@app.entrypoint
async def invoke(payload: dict, context):
    """Runtime entrypoint: handle one customer message and stream events.

    Args:
        payload: Request body, `{"message": "...", "customer_id": "cust001"}`.
        context: Runtime request context; `context.session_id` is the conversation id.

    Yields:
        dict: Events: `agent_selected`, `progress`, `reply`, then `done`.
    """
    message = str(payload.get("message", ""))
    customer_id = str(payload.get("customer_id", "cust001"))
    agent = route(message)
    logger.info("session=%s agent=%s", getattr(context, "session_id", ""), agent)
    yield {"type": "agent_selected", "agent": agent}

    if agent == "order_management":
        yield {"type": "reply", "agent": agent, "text": await run_order_agent(message, customer_id)}
    elif agent == "shipment_tracking":
        reply = ""
        async for kind, text in run_shipment_agent(message, customer_id):
            if kind == "progress":
                yield {"type": "progress", "agent": agent, "text": text}
            else:
                reply += text
        yield {"type": "reply", "agent": agent, "text": reply}
    else:
        yield {"type": "reply", "agent": agent, "text": "I can help with orders and parcel tracking."}
    yield {"type": "done"}


if __name__ == "__main__":
    app.run(port=8090)  # Runtime itself uses 8080; 8090 here avoids clashing with other local apps

## 5.2 Start it and check the health endpoint

In [ ]:
start_service("supervisor", "supervisor_runtime.py", port=8090)
show("GET /ping ->", httpx.get("http://127.0.0.1:8090/ping").json())

## 5.3 Invoke it like Runtime would

We `POST /invocations` and read the streamed response line by line. Runtime sends **Server-Sent Events**: every
line looks like `data: {...json...}`. The session id header is how Runtime ties requests to one conversation.

First an **order question** (goes: supervisor -> MCP server).

In [ ]:
SESSION_HEADERS = {"X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": "demo-session-0000000000000000000000001"}


def invoke_runtime(message: str, customer_id: str = "cust001") -> None:
    """POST a message to the local Runtime app and log each streamed event as it arrives.

    Args:
        message: The customer's message.
        customer_id: Customer id sent in the body (local mode; in AWS it comes from the login token).
    """
    body = {"message": message, "customer_id": customer_id}
    with httpx.stream("POST", "http://127.0.0.1:8090/invocations", json=body, headers=SESSION_HEADERS, timeout=60) as response:
        for line in response.iter_lines():
            if line.startswith("data:"):
                event = json.loads(line.removeprefix("data:").strip())
                show(f"  {event.get('type'):15}", event.get("agent", "") + " " + event.get("text", ""))


show("Q: What was my last purchase?")
invoke_runtime("What was my last purchase?")

Now a **shipping question** (supervisor -> A2A shipment agent). Watch the `progress` events arrive one second apart, that is A2A streaming passing straight through the supervisor to you.

In [ ]:
show("Q: Where is my order ORD-1002?")
invoke_runtime("Where is my order ORD-1002?")

And something **out of scope** (nobody is called).

In [ ]:
show("Q: What is the weather?")
invoke_runtime("What is the weather like?")

### Recap of what just happened

```
notebook --POST /invocations--> supervisor (Runtime app)
                                   |-- route() picks an agent
                                   |-- order question   --MCP  call_tool-->  order MCP server
                                   '-- shipping question --A2A  SendMessage--> shipment agent (streams progress)
```

Locally the supervisor and agents talk over plain HTTP. In AWS, Runtime adds **authentication** (IAM/SigV4 or a JWT),
session isolation and scaling around exactly this code.

# Part 6. AgentCore Gateway (needs AWS)

## What problem does Gateway solve?

Right now the order agent knows the MCP server's address. With many tools, teams and APIs that gets messy:
every agent needs every URL and its own auth. **Gateway** is one managed MCP endpoint in front of all of it:

```
  agent --(one URL + one token)--> AgentCore GATEWAY --> target 1: your MCP server
                                      |                  target 2: a Lambda function
                                      |                  target 3: an OpenAPI / REST API
                                      '-- handles inbound auth, outbound credentials, tool discovery
```

- The agent sees a **single MCP server**; `list_tools()` returns the tools of *all* targets.
- **Inbound auth**: who may call the Gateway (IAM, or a JWT from an identity provider such as Auth0/Cognito).
- **Outbound auth**: how Gateway authenticates to each target (API key, OAuth, IAM role).

## How it plugs into our code

You already wrote the agent side: `order_agent(..., mcp_url=...)`. With Gateway, **nothing in the agent changes except**
the URL (and a bearer token):

```
ORDER_MCP_URL   = https://<gateway-id>.gateway.bedrock-agentcore.<region>.amazonaws.com/mcp
ORDER_MCP_TOKEN = <access token from your identity provider>
```

One catch: Gateway runs in AWS, so the target MCP server must be **reachable from AWS**. `127.0.0.1:8001` is not.
The usual path is to deploy the MCP server itself to AgentCore Runtime with the **MCP protocol** (port 8000, path `/mcp`,
stateless: `FastMCP(..., host="0.0.0.0", stateless_http=True)`), then register that Runtime URL as the Gateway target.

## 6.1 The AWS steps, in order (guarded, off by default)

1. Build and deploy the MCP server as a Runtime (ARM64 container image in ECR, then `create_agent_runtime` with protocol `MCP`).
2. Create an IAM role that Gateway may assume.
3. `create_gateway(...)` with inbound auth.
4. `create_gateway_target(...)` pointing at the MCP server.
5. Set `ORDER_MCP_URL` / `ORDER_MCP_TOKEN` and restart the supervisor.

> **Cost warning:** these create billable AWS resources. Gateway and Runtime are pay-per-use, but only run this after
> Parts 1-5 work, use the smallest settings, and delete the resources afterwards. Exact parameter names evolve, so check
> the current `bedrock-agentcore-control` boto3 docs before running.

In [ ]:
RUN_AWS = False                 # <-- flip to True only when you really want to create AWS resources
AWS_REGION = "us-east-1"
GATEWAY_ROLE_ARN = "arn:aws:iam::<account-id>:role/<gateway-role>"       # step 2
MCP_SERVER_URL = "https://<your-mcp-runtime-or-public-url>/mcp"          # step 1
JWT_DISCOVERY_URL = "https://<your-tenant>/.well-known/openid-configuration"
JWT_ALLOWED_AUDIENCE = ["https://hm-agent/api"]


def create_gateway_with_mcp_target() -> dict:
    """Create a Gateway (JWT inbound auth) with the order MCP server as its target.

    Returns:
        dict: `{"gateway_url": ..., "gateway_id": ...}`; put the URL into `ORDER_MCP_URL`.
    """
    import boto3

    control = boto3.client("bedrock-agentcore-control", region_name=AWS_REGION)
    gateway = control.create_gateway(
        name="hm-demo-gateway",
        roleArn=GATEWAY_ROLE_ARN,
        protocolType="MCP",
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={"customJWTAuthorizer": {
            "discoveryUrl": JWT_DISCOVERY_URL, "allowedAudience": JWT_ALLOWED_AUDIENCE}},
    )
    control.create_gateway_target(
        gatewayIdentifier=gateway["gatewayId"],
        name="order-management",
        targetConfiguration={"mcp": {"mcpServer": {"endpoint": MCP_SERVER_URL}}},
        credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
    )
    return {"gateway_url": gateway["gatewayUrl"], "gateway_id": gateway["gatewayId"]}


if RUN_AWS:
    show("created:", create_gateway_with_mcp_target())
else:
    show("RUN_AWS is False -> nothing was created. Read the function above to see the calls.")

## 6.2 Try the "swap the URL" idea locally

We cannot start a real Gateway offline, but we can prove the claim "only the URL changes". The supervisor already reads
`ORDER_MCP_URL` and `ORDER_MCP_TOKEN` from the environment. Restarting it with different values is all that Gateway needs.
Here we restart it with the same local URL, just to show the mechanism.

In [ ]:
start_service("supervisor", "supervisor_runtime.py", port=8090,
              env={"ORDER_MCP_URL": "http://127.0.0.1:8001/mcp", "ORDER_MCP_TOKEN": ""})
invoke_runtime("Show my orders")

# Part 7. Clean up and what to learn next

## 7.1 Stop the background processes

In [ ]:
for name in list(processes):
    stop_service(name)
show("stopped all services")

## 7.2 How this maps to the real project

| This notebook | Real project (`src/hm_agent/`) |
|---|---|
| `order_mcp_server.py` (dict data) | `order_management/mcp_server.py` (Neon Postgres, `customer_id` injected by the client) |
| `choose_order_tool` rules | an LLM `create_agent` ReAct loop |
| `ShipmentExecutor` | `a2a/server.py` generic executor + `shipment_tracking/` |
| `route()` keywords | `supervisor/agent.py` structured-output LLM + LangGraph |
| `supervisor_runtime.py` | `runtime/entrypoint.py` (auth, thread ownership, history, tracing) |
| no auth | JWT verification (`auth/`) and SigV4 signing (`a2a/sigv4.py`) |

## 7.3 Suggested next steps

1. Replace `choose_order_tool` with an LLM that receives the MCP tool schemas (see `langchain-mcp-adapters`).
2. Add a bearer-token check to the MCP server, then send it as `ORDER_MCP_TOKEN` to feel how Gateway auth works.
3. Make the supervisor's `route()` an LLM call (cheap model, structured output).
4. Deploy **one** piece (the MCP server) to AgentCore Runtime, then put Gateway in front of it (Part 6).